# Try microns-finder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsanzros/microns-neuroglancer-finder/blob/main/notebooks/try_microns_finder.ipynb)

Find cells in the [MICrONS](https://www.microns-explorer.org/cortical-mm3) cubic millimeter of mouse visual cortex by asking for them in plain words, and open them in Neuroglancer.

**No installation and no account needed.** Run the cells from top to bottom (Shift+Enter). The first one takes about 30 seconds.

Without a CAVE token, the tool uses a prebuilt copy of the MICrONS cell table (release v1926, 144,120 nuclei). That's enough to search cells by type, layer, area and proofreading. A free token, covered at the end, adds synapses and shorter links with labels.

In [1]:
# Install the tool. This skips the extra packages of its AI (MCP) server, which this notebook doesn't need.
%pip install -q caveclient nglui pandas pyarrow pyyaml
%pip install -q --no-deps "git+https://github.com/jsanzros/microns-neuroglancer-finder@v0.1.2"

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import HTML, display
from microns_finder import Finder, service
from microns_finder.finder import NO_TOKEN_NOTE

f = Finder()  # without a token this downloads the prebuilt cell table (8 MB)
print(f"{len(f.cells):,} cells loaded from MICrONS release v{f.version}")


def show(request, **options):
    """Find cells, show them as a table and a Neuroglancer link."""
    options.setdefault("n", 3)
    p = service.find(f, request, **options)
    if "error" in p:
        print(p["error"])
        return
    print(f"{p['pool_size']:,} cells match; showing {p['n_returned']}.")
    table = pd.DataFrame(p["cells"])[["nucleus_id", "subclass", "mtype", "meso_type", "layer", "depth_um", "visual_area", "proofreading"]]
    display(table)
    display(HTML(f'<p><a href="{p["link"]}" target="_blank"><b>Open these cells in Neuroglancer ↗</b></a></p>'))
    for note in p["notes"]:
        if note != NO_TOKEN_NOTE:  # explained at the top
            print("note:", note)

144,120 cells loaded from MICrONS release v1926


## Ask for cells

Each request returns three example cells by default, and a link that opens them in Neuroglancer with each cell colored. Click a soma point in the `somata` layer to see its labels, depth and proofreading.

The first time you open a link, Neuroglancer asks you to **sign in with Google**. It's free; MICrONS uses it to serve the segmentation.

In [3]:
show("astrocyte in layer 1")

785 cells match; showing 3.


,nucleus_id,subclass,mtype,meso_type,layer,depth_um,visual_area,proofreading
0,255123,astrocyte,None,None,L1,52.1,V1,glia clean
1,291332,astrocyte,None,None,L1,54.7,V1,glia clean
2,255185,astrocyte,None,None,L1,43.4,V1,glia clean


In [4]:
show("oligodendrocyte in the white matter")

256 cells match; showing 3.


,nucleus_id,subclass,mtype,meso_type,layer,depth_um,visual_area,proofreading
0,139325,oligo,None,None,WM,741.7,V1,none
1,176972,oligo,None,None,WM,748.6,V1,none
2,176941,oligo,None,None,WM,745.7,V1,none


note: 92% of the 12841 nuclei in the white matter have no cell-type label, so white-matter results come from the small labeled subset.


MICrONS has three cell-type classifications, and they don't always agree. `labels="consensus"` keeps only cells that every classification calls the requested type:

In [5]:
show("neurogliaform cell", labels="consensus")

163 cells match; showing 3.


,nucleus_id,subclass,mtype,meso_type,layer,depth_um,visual_area,proofreading
0,291201,NGC,STC,NGC,L1,50.1,V1,"dendrite extended, axon partially_extended"
1,294561,NGC,STC,NGC,L3,189.5,V1,"dendrite extended, axon partially_extended"
2,262779,NGC,STC,NGC,L4,339.9,V1,"dendrite extended, axon partially_extended"


In [6]:
show("chandelier cell")

29 cells match; showing 3.


,nucleus_id,subclass,mtype,meso_type,layer,depth_um,visual_area,proofreading
0,255275,NGC,DTC,ChC,L1,79.1,V1,"dendrite extended, axon fully_extended"
1,152705,BC,PTC,ChC,L2,93.4,V1,none
2,516594,BC,PTC,ChC,L2,131.1,AL,none


note: 'chandelier cell (axo-axonic)' has no cell_type definition; used meso_type.
note: Only the multifeature model labels chandelier cells (a few dozen in the volume).


Change `seed` to get other examples, or `n` for more cells. `proofread` keeps only manually proofread cells:

In [7]:
show("basket cell in layer 4", seed=1)

675 cells match; showing 3.


,nucleus_id,subclass,mtype,meso_type,layer,depth_um,visual_area,proofreading
0,333180,BC,PTC,PV,L4,316.3,V1,"dendrite extended, axon fully_extended"
1,299317,BC,PTC,PV,L4,317.0,V1,"dendrite extended, axon fully_extended"
2,299222,BC,PTC,PV,L4,339.5,V1,"dendrite extended, axon fully_extended"


note: Includes PV-like and VIP/Sncg-like ("AltBasket") basket cells; use "PV basket" or "CCK basket" for one of them.


In [8]:
show("L5 ET", area="V1", proofread="full", n=1)

32 cells match; showing 1.


,nucleus_id,subclass,mtype,meso_type,layer,depth_um,visual_area,proofreading
0,266839,5P-ET,L5ET,L5ET,L5,474.5,V1,"dendrite extended, axon fully_extended"


## Your turn

Edit the request and run the cell. Plain names work, such as "microglia", "Martinotti cell", "L2/3 pyramidal", "VIP interneuron", "pericyte" or "L6 corticothalamic". So do raw MICrONS labels like `23P` or `L4c`. The next cell lists every cell type the tool knows.

In [9]:
show("microglia")

2,359 cells match; showing 3.


,nucleus_id,subclass,mtype,meso_type,layer,depth_um,visual_area,proofreading
0,260696,microglia,None,None,L4,289.2,V1,glia clean
1,296791,microglia,None,None,L4,251.8,V1,glia clean
2,301194,microglia,None,None,L5,424.6,V1,glia clean


In [10]:
pd.DataFrame(service.terms())[["key", "label", "synonyms"]]

,key,label,synonyms
0,astrocyte,astrocyte,"[astrocytes, astro, astros, astroglia, astrogl..."
1,oligodendrocyte,oligodendrocyte,"[oligodendrocytes, oligo, oligos, oligodendrog..."
2,opc,oligodendrocyte precursor cell (OPC),"[OPC, OPCs, oligodendrocyte precursor, oligode..."
3,microglia,microglia,"[microglial cell, microglial cells, microglias..."
4,pericyte,pericyte,"[pericytes, mural cell, mural cells]"
5,glia,"glial cell (astrocyte, oligodendrocyte, OPC or...","[glial, glial cell, glial cells, neuroglia, ne..."
6,nonneuronal,any non-neuronal cell,"[non-neuronal, non-neuronal cell, nonneuronal ..."
7,neuron,any neuron,"[neurons, nerve cell]"
8,excitatory,excitatory neuron (pyramidal),"[excitatory neuron, excitatory neurons, excita..."
9,inhibitory,inhibitory neuron (interneuron),"[interneuron, interneurons, inhibitory neuron,..."


## Optional: add a free CAVE token

A token unlocks synapses (e.g. "basket cells onto L2/3 pyramidal cells"), shorter links with cell labels, and other MICrONS releases.

1. Open https://global.daf-apis.com/auth/api/v1/create_token, sign in with Google, accept the MICrONS terms if asked, and copy the token.
2. In Colab, click the **key icon** (Secrets) in the left sidebar, add a secret named `CAVE_TOKEN` with the token as its value, and allow this notebook to use it.
3. Run the next two cells.

Keep the token private: it's like a password for your MICrONS access. Secrets keep it out of the notebook itself.

In [11]:
from caveclient import CAVEclient

try:
    from google.colab import userdata
    token = userdata.get("CAVE_TOKEN")
except Exception:  # not in Colab, or no secret yet
    token = None

if token:
    CAVEclient().auth.save_token(token=token, overwrite=True)
    f = Finder()
    print("Token saved. Synapses and short links are available:", f.online)
else:
    print("No CAVE_TOKEN secret found, so the notebook continues without a token.")

No CAVE_TOKEN secret found, so the notebook continues without a token.


In [12]:
if f.online:
    s = service.synapses(f, "basket cell onto L2/3 pyramidal", n=1)
    for pair in s.get("pairs", []):
        print(f"{pair['n_synapses']} synapses from nucleus {pair['pre']['nucleus_id']} onto nucleus {pair['post']['nucleus_id']}")
        display(HTML(f'<p><a href="{pair["link"]}" target="_blank"><b>Open both cells and their synapses ↗</b></a></p>'))
    for note in s.get("notes", []):
        print("note:", note)
    if "error" in s:
        print(s["error"])
else:
    print("Add a token above to try synapses.")

Add a token above to try synapses.


## Next steps

- **Use it with your AI assistant:** ChatGPT, Gemini, Claude, Codex, Copilot or Cursor can answer requests like "show me a microglia" with real cells through the tool's MCP server. See the [README](https://github.com/jsanzros/microns-neuroglancer-finder#connect-your-ai).
- **Install it on your computer:** `pip install "git+https://github.com/jsanzros/microns-neuroglancer-finder"`.
- **Cite it:** [doi.org/10.5281/zenodo.23326398](https://doi.org/10.5281/zenodo.23326398), together with the MICrONS papers listed in the README.

The cell table is derived from the MICrONS data (MICrONS Consortium, *Nature* 2025), released under CC BY 4.0.